# Assignment 2 — Part B: Word Embeddings
**Member:** Hadia (Roll No. 1139)  
**Dataset:** Assignment 1 cleaned ToxicChat training split  
**Models:** Word2Vec CBOW, Word2Vec Skip-gram, fastText

Run all cells from top to bottom after installing dependencies. **No results are claimed until executed.**

## B0. Setup and data integrity
Reuse `data/processed/toxicchat_train_cleaned.csv`; do not clean the dataset differently. Train on the **training split only**. Models are saved in `models/` (generated binaries are not required in Git if reproducible scripts are provided).

In [1]:
from pathlib import Path
import sys, random, re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText
from gensim.utils import simple_preprocess

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed' / 'toxicchat_train_cleaned.csv').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed' / 'toxicchat_train_cleaned.csv'
assert DATA.exists(), f'Missing cleaned training data: {DATA}'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0])
print('Training CSV:', DATA)
print('Model directory:', MODEL_DIR)

Python: 3.11.9
Training CSV: f:\7th sem\NLP\Project\Hadia\A2\data\processed\toxicchat_train_cleaned.csv
Model directory: f:\7th sem\NLP\Project\Hadia\A2\models


In [2]:
df = pd.read_csv(DATA)
assert {'cleaned_text', 'toxicity'}.issubset(df.columns), df.columns.tolist()
texts = df['cleaned_text'].fillna('').astype(str).tolist()
sentences = [simple_preprocess(text, deacc=False, min_len=1) for text in texts]
sentences = [tokens for tokens in sentences if tokens]
assert sentences, 'No usable training documents'
print('Training rows:', len(df))
print('Non-empty tokenized documents:', len(sentences))
print('Sample tokens:', sentences[0][:25])

Training rows: 5082
Non-empty tokenized documents: 5081
Sample tokens: ['masturbacja', 'jest', 'proces', 'co', 'oitrzebuje']


## B1–B2. CBOW and Skip-gram Word2Vec
Use identical hyperparameters to make the objective (`sg`) the main controlled difference. `workers=1` supports reproducible runs, though numerical results can vary by library/platform.

In [3]:
PARAMS = dict(vector_size=100, window=5, min_count=2, epochs=20, negative=5, seed=SEED, workers=1)
print('Shared hyperparameters:', PARAMS)
cbow = Word2Vec(sentences=sentences, sg=0, **PARAMS)
skipgram = Word2Vec(sentences=sentences, sg=1, **PARAMS)
print('CBOW vocabulary:', len(cbow.wv), 'dimensions:', cbow.vector_size)
print('Skip-gram vocabulary:', len(skipgram.wv), 'dimensions:', skipgram.vector_size)
assert cbow.vector_size == skipgram.vector_size == PARAMS['vector_size']
assert set(cbow.wv.key_to_index) == set(skipgram.wv.key_to_index)

Shared hyperparameters: {'vector_size': 100, 'window': 5, 'min_count': 2, 'epochs': 20, 'negative': 5, 'seed': 42, 'workers': 1}
CBOW vocabulary: 6375 dimensions: 100
Skip-gram vocabulary: 6375 dimensions: 100


## B3. Comparison of learning objectives
- **CBOW (`sg=0`)**: predicts a centre word from surrounding context words; typically efficient for frequent patterns.
- **Skip-gram (`sg=1`)**: predicts surrounding context words from a centre word; can capture useful relationships for less frequent words, but is not guaranteed to outperform CBOW.

Compare the actual nearest neighbours below; model quality cannot be inferred from the objective alone.

In [4]:
def show_neighbours(model, query, n=5):
    if query not in model.wv:
        print(f'{query!r}: OOV for {type(model).__name__}')
        return []
    results = model.wv.most_similar(query, topn=n)
    print(f'{query!r}:', [(w, round(float(s), 4)) for w, s in results])
    return results

candidates = ['toxic', 'hate', 'help', 'write', 'people', 'you']
queries = [word for word in candidates if word in cbow.wv][:4]
if not queries:
    queries = list(cbow.wv.index_to_key[:4])
print('Queries:', queries)
for q in queries:
    print('\nCBOW:'); show_neighbours(cbow, q)
    print('Skip-gram:'); show_neighbours(skipgram, q)

Queries: ['hate', 'help', 'write', 'people']

CBOW:
'hate': [('therefore', 0.8852), ('ace', 0.8796), ('difficult', 0.8795), ('sometimes', 0.8662), ('spp', 0.8659)]
Skip-gram:
'hate': [('racism', 0.7835), ('restrict', 0.7455), ('claim', 0.7363), ('absolutely', 0.7208), ('swear', 0.7205)]

CBOW:
'help': [('teach', 0.7316), ('recommend', 0.7232), ('assist', 0.7015), ('chat', 0.6752), ('build', 0.6577)]
Skip-gram:
'help': [('assist', 0.5662), ('modular', 0.5617), ('mental', 0.5511), ('creator', 0.5471), ('subtopics', 0.5425)]

CBOW:
'write': [('create', 0.8059), ('compose', 0.7649), ('generate', 0.7306), ('teller', 0.7249), ('short', 0.6958)]
Skip-gram:
'write': [('compose', 0.6655), ('wirte', 0.6397), ('generate', 0.6278), ('programmer', 0.6187), ('formal', 0.6042)]

CBOW:
'people': [('they', 0.7637), ('thing', 0.7587), ('fat', 0.7364), ('much', 0.7274), ('weight', 0.7191)]
Skip-gram:
'people': [('stereotype', 0.6362), ('evil', 0.6262), ('unusual', 0.6149), ('scent', 0.6105), ('themselves

## B4. fastText subword embeddings
Gensim fastText uses character n-grams to form representations for many words not present as complete training vocabulary entries. For OOV words, test with `get_vector()` rather than `word in model.wv` (the latter checks known vocabulary).

In [5]:
fasttext = FastText(sentences=sentences, sg=1, min_n=3, max_n=6, **PARAMS)
print('fastText vocabulary:', len(fasttext.wv), 'dimensions:', fasttext.vector_size)
known = queries[0]
misspelled = known + 'x'
rare = next((w for w in fasttext.wv.index_to_key if w not in fasttext.wv.index_to_key[:100] and len(w) >= 5), known)
for token in [known, rare, misspelled, 'unseenwordxyz']:
    print('\nToken:', repr(token), '| in trained vocabulary:', token in fasttext.wv.key_to_index)
    for name, model in [('CBOW', cbow), ('Skip-gram', skipgram), ('fastText', fasttext)]:
        if name != 'fastText' and token not in model.wv:
            print(name, ': no vector (OOV)')
        else:
            try:
                vec = model.wv.get_vector(token)
                print(name, ': vector available, shape', vec.shape, 'norm', round(float(np.linalg.norm(vec)), 4))
            except KeyError:
                print(name, ': no vector (OOV)')
print('\nfastText nearest neighbours for known token:')
show_neighbours(fasttext, known)

fastText vocabulary: 6375 dimensions: 100

Token: 'hate' | in trained vocabulary: True
CBOW : vector available, shape (100,) norm 1.4188
Skip-gram : vector available, shape (100,) norm 2.1403
fastText : vector available, shape (100,) norm 3.586

Token: 'example' | in trained vocabulary: True
CBOW : vector available, shape (100,) norm 7.8208
Skip-gram : vector available, shape (100,) norm 3.9024
fastText : vector available, shape (100,) norm 4.4333

Token: 'hatex' | in trained vocabulary: False
CBOW : no vector (OOV)
Skip-gram : no vector (OOV)
fastText : vector available, shape (100,) norm 2.5841

Token: 'unseenwordxyz' | in trained vocabulary: False
CBOW : no vector (OOV)
Skip-gram : no vector (OOV)
fastText : vector available, shape (100,) norm 1.1325

fastText nearest neighbours for known token:
'hate': [('kate', 0.8601), ('affiliate', 0.8227), ('gate', 0.8009), ('debate', 0.7978), ('escalate', 0.7935)]


[('kate', 0.8601344227790833),
 ('affiliate', 0.8226880431175232),
 ('gate', 0.8009381294250488),
 ('debate', 0.7977897524833679),
 ('escalate', 0.7934607863426208)]

## Model persistence and reloading
Save the models and confirm that they can be reloaded without retraining. The notebook itself is a reproducible regeneration procedure for Part D.

In [6]:
paths = {'cbow': MODEL_DIR / 'word2vec_cbow.model', 'skipgram': MODEL_DIR / 'word2vec_skipgram.model', 'fasttext': MODEL_DIR / 'fasttext.model'}
cbow.save(str(paths['cbow']))
skipgram.save(str(paths['skipgram']))
fasttext.save(str(paths['fasttext']))
loaded_cbow = Word2Vec.load(str(paths['cbow']))
loaded_skipgram = Word2Vec.load(str(paths['skipgram']))
loaded_fasttext = FastText.load(str(paths['fasttext']))
assert np.allclose(loaded_cbow.wv[known], cbow.wv[known])
assert np.allclose(loaded_skipgram.wv[known], skipgram.wv[known])
assert np.allclose(loaded_fasttext.wv.get_vector(misspelled), fasttext.wv.get_vector(misspelled))
for name, path in paths.items():
    print(name, 'saved and reload-verified:', path.relative_to(ROOT))
print('All model reload checks passed.')

cbow saved and reload-verified: models\word2vec_cbow.model
skipgram saved and reload-verified: models\word2vec_skipgram.model
fasttext saved and reload-verified: models\fasttext.model
All model reload checks passed.


## Part B deliverables and interpretation
1. **B1:** CBOW model with documented parameters, vocabulary size, neighbours.
2. **B2:** Skip-gram model using the same parameters.
3. **B3:** CBOW context→centre vs Skip-gram centre→context explanation, plus observed comparisons.
4. **B4:** fastText with subword settings and known/rare/misspelled/OOV demonstrations.
5. Models saved and reload-verified in `models/` when notebook runs.

**For final submission:** Run All, inspect outputs, save executed notebook, and push it. Record actual results in the team PDF. Never claim this unexecuted GitHub notebook contains empirical results.